# Bank Marketing Campaign — Data Cleaning & Exploratory Data Analysis



---

## 1.Imports



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder


## 2. Loading the Dataset



In [ ]:
df = pd.read_csv('bank-full.csv', sep=';')
# The target column in bank-full.csv is 'y' — we rename it to 'deposit' for consistency
df = df.rename(columns={'y': 'deposit'})
print(f'Loaded {df.shape[0]} rows and {df.shape[1]} columns')
df.head(10)

## 3. Data Inspection


In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
df.describe(include='object')

### 3.1 Missing Values & Duplicates



In [ ]:
missing = df.isnull().sum()
missing_df = pd.DataFrame({'Missing Count': missing})
print(missing_df)
print(f'\nTotal NaN values in the dataset: {df.isnull().sum().sum()}')

In [ ]:
dup_count = df.duplicated().sum()
print(f'Duplicate rows found: {dup_count} ({dup_count/len(df)*100:.2f}%)')

In [ ]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    unknown_count = (df[col] == 'unknown').sum()
    if unknown_count > 0:
        print(f'  {col}: {unknown_count} unknowns ')

## 4. Data Cleaning

.

In [ ]:
initial_rows = len(df)
df.drop_duplicates(inplace=True)
removed = initial_rows - len(df)
print(f'Removed {removed} duplicate rows ')

### 4.2 Dealing with 'Unknown' Values



In [ ]:
for col in ['job', 'education']:
    mode_val = df[df[col] != 'unknown'][col].mode()[0]
    unknown_ct = (df[col] == 'unknown').sum()
    df[col] = df[col].replace('unknown', mode_val)
    print(f"{col}: filled {unknown_ct} unknowns with '{mode_val}' (most common value)")


### 4.3 Data Type Fixes & Target Encoding



In [ ]:
df['deposit_binary'] = df['deposit'].map({'yes': 1, 'no': 0})

month_order = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
               'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
df['month_num'] = df['month'].map(month_order)
print(f'\nData types after changes:\n{df.dtypes}')

### 4.4 Cleaning Up Categorical Values



In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()

df['job'] = df['job'].replace('admin.', 'admin')


### 5.2 Visualizing Outliers



In [ ]:
num_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for i, col in enumerate(num_cols):
    ax = axes[i//3, i%3]
    bp = ax.boxplot(df[col].dropna(), patch_artist=True, vert=True)
    bp['boxes'][0].set_facecolor(colors[i])
    bp['boxes'][0].set_alpha(0.7)
    ax.set_title(f'{col}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Box Plots —before removing Outliers', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.3 Treating Outliers — Capping



In [ ]:
# Cap outliers using IQR method (excluding pdays which has special meaning: -1 = not contacted)
cap_cols = ['age', 'balance', 'duration', 'campaign', 'previous']

for col in cap_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    after = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'{col}: capped {before} outliers (lower={lower:.0f}, upper={upper:.0f})')

print('\n✅ Outlier treatment complete (pdays excluded — special sentinel value -1).')

In [ ]:
num_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for i, col in enumerate(num_cols):
    ax = axes[i//3, i%3]
    bp = ax.boxplot(df[col].dropna(), patch_artist=True, vert=True)
    bp['boxes'][0].set_facecolor(colors[i])
    bp['boxes'][0].set_alpha(0.7)
    ax.set_title(f'{col}', fontsize=14, fontweight='bold')
    ax.set_ylabel('Value')

plt.suptitle('Box Plots After removing Outliers', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.2 Encoding Categorical Variables



In [ ]:
le = LabelEncoder()
binary_cols = ['default', 'housing', 'loan', 'deposit']

df_encoded = df.copy()
for col in binary_cols:
    df_encoded[col + '_enc'] = le.fit_transform(df_encoded[col])
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

nominal_cols = ['job', 'marital', 'contact', 'poutcome', 'month']
df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=True, dtype=int)

print(f'\nEncoding done. Columns went from {df.shape[1]} to {df_encoded.shape[1]}.')

## 7. Exploratory Data Analysis



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_target = ['#e74c3c', '#2ecc71']
counts = df['deposit'].value_counts()

# Bar chart
axes[0].bar(counts.index, counts.values, color=colors_target, edgecolor='black', alpha=0.85)
axes[0].set_title('Term Deposit — Yes vs No', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Subscribed?')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart for proportions
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors_target,
            startangle=90, explode=[0.05, 0], shadow=True, textprops={'fontsize': 13})
axes[1].set_title('Subscription Split', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'Counts:\n{counts}')

### 7.2 Numerical Feature Distributions



In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for i, col in enumerate(num_cols):
    ax = axes[i//3, i%3]
    ax.hist(df[col], bins=30, color=colors[i], edgecolor='black', alpha=0.75)
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='blue', linestyle='--', linewidth=2, label=f'Median: {df[col].median():.1f}')
    ax.set_title(f'{col}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Distribution of Numerical Features', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Save Cleaned Dataset



In [ ]:
cols_to_drop = ['age_group', 'balance_category', 'duration_category', 'deposit', 'month']
df_clean = df_encoded.drop(columns=[c for c in cols_to_drop if c in df_encoded.columns], errors='ignore')

for c in ['age_group', 'balance_category', 'duration_category']:
    if c in df_clean.columns:
        df_clean.drop(columns=[c], inplace=True)

df_clean.to_csv('cleaned_data.csv', index=False)
print(f'Saved cleaned_data.csv — {df_clean.shape[0]} rows, {df_clean.shape[1]} columns')
df_clean.head()